# fine-tune-forge — QLoRA Fine-tuning Pipeline

**Prima di eseguire**: `Runtime → Change runtime type → T4 GPU`

**Secrets richiesti** (icona 🔑, tutti con Notebook access ON):
| Nome secret | Obbligatorio | Cosa è |
|---|---|---|
| `GROQ_API_KEY` | ✅ Sì | Groq API key — [console.groq.com](https://console.groq.com) (free) |
| `HF_TOKEN` | ⚠️ Solo per export | HuggingFace token **Write** — [crea qui](https://huggingface.co/settings/tokens) → *New token* → Type: **Write** |
| `HF_REPO` | ⚠️ Solo per export | Nome repo HF output, es. `MatteoAdamo82/agente-ristorante` |
| `GEMINI_API_KEY` | ❌ Solo se usi `--generator gemini` | Google AI Studio key |
| `GITHUB_TOKEN_COLAB` | ❌ No (repo pubblico) | Solo se il repo è privato |

## 1. Setup

In [ ]:
# Controlla GPU
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print(f'✓ GPU: {r.stdout.strip()}' if r.returncode == 0 else '⚠️  Nessuna GPU — Runtime → Change runtime type → T4 GPU')

In [ ]:
# Clona il repo (o aggiorna se già presente)
import os, subprocess, shutil

PUBLIC_REPO = 'https://github.com/MatteoAdamo82/fine-tuning.git'
DEST = '/content/fine-tuning'

repo_url = PUBLIC_REPO
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN_COLAB')
    if token:
        repo_url = f'https://{token}@github.com/MatteoAdamo82/fine-tuning.git'
        print('ℹ️  GITHUB_TOKEN_COLAB trovato — clone con autenticazione')
    else:
        print('✓ Clone pubblico (nessun token)')
except Exception:
    print('✓ Clone pubblico (nessun token)')

if os.path.exists(DEST):
    # Già presente: aggiorna invece di ri-clonare
    print(f'ℹ️  {DEST} già esistente — git pull...')
    r = subprocess.run(['git', '-C', DEST, 'pull', '--rebase'], capture_output=True, text=True)
    if r.returncode != 0:
        print(f'⚠️  git pull fallito, ri-clono da zero...')
        shutil.rmtree(DEST)
        r = subprocess.run(['git', 'clone', '-q', repo_url, DEST], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f'Clone fallito:\n{r.stderr}')
    print(f'✓ Repo aggiornato')
else:
    r = subprocess.run(['git', 'clone', '-q', repo_url, DEST], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'Clone fallito:\n{r.stderr}')
    print('✓ Repo clonato')

os.chdir(DEST)
print(f'✓ Working dir: {DEST}')

In [ ]:
# Installa dipendenze
!pip install -q -e . 2>&1 | tail -3

# Verifica che il CLI funzioni
r = subprocess.run(['python', '-m', 'src.cli', '--help'], capture_output=True, text=True)
print('✓ CLI installata' if r.returncode == 0 else f'⚠️  Errore CLI: {r.stderr}')

## 2. Configurazione

In [ ]:
# Carica API keys dai Secrets
import os
from google.colab import userdata

# Groq (default generator) — https://console.groq.com per la chiave
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
print('✓ GROQ_API_KEY caricata')

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('✓ HF_TOKEN caricato')
except Exception:
    print('ℹ️  HF_TOKEN non trovato — export HF Hub disabilitato')

try:
    HF_REPO = userdata.get('HF_REPO')
    print(f'✓ HF_REPO: {HF_REPO}')
except Exception:
    HF_REPO = ''
    print('ℹ️  HF_REPO non trovato — imposta qui sotto se vuoi salvare il modello')

# Gemini opzionale (solo se usi --generator gemini)
try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('✓ GEMINI_API_KEY caricata (opzionale)')
except Exception:
    pass

In [ ]:
# ── Parametri — modifica qui ───────────────────────────────────────────────────
DOMINIO  = 'restaurant_booking'  # restaurant_booking | sales_agent | professional_booking
MODELLO  = 'Qwen/Qwen3-0.6B'    # Qwen/Qwen3-0.6B (~2GB VRAM) | Qwen/Qwen3-4B (~6GB)
N_ESEMPI = 50                    # 50 per test, 150 per produzione
# HF_REPO = 'MatteoAdamo82/agente-ristorante'  # decommenta se non lo hai nei Secrets
# ──────────────────────────────────────────────────────────────────────────────

DATASET_PATH = f'data/processed/{DOMINIO}.jsonl'
print(f'Dominio: {DOMINIO} | Modello: {MODELLO} | Esempi: {N_ESEMPI}')
print(f'HF_REPO: {HF_REPO or "NON IMPOSTATO — il modello non verrà salvato"}')

## 3. Verifica installazione

In [ ]:
!python -m src.cli list-domains

## 4. Generazione Dataset

Il dataset viene generato una sola volta e salvato in `data/processed/{dominio}.jsonl`.
Se esiste già, viene **riusato automaticamente** — nessun API call sprecato.

- `--force-dataset` per rigenerare da zero
- `--generator ollama` per usare un modello locale (no API key)
- Puoi **editare manualmente** il JSONL prima di avviare il training (cella successiva)

In [ ]:
# Genera il dataset (riusato automaticamente se già presente)
# Per forzare rigenerazione: aggiungi --force-dataset
# Per usare Ollama locale: aggiungi --generator ollama
!python -m src.cli dataset \
    --dominio {DOMINIO} \
    --n-examples {N_ESEMPI} \
    --output {DATASET_PATH}

In [ ]:
# Anteprima dataset — controlla qualità prima di avviare il training
import json
from pathlib import Path

if not Path(DATASET_PATH).exists():
    raise FileNotFoundError(f'Dataset non trovato: {DATASET_PATH}')

with open(DATASET_PATH) as f:
    esempi = [json.loads(l) for l in f if l.strip()]

print(f'✓ Esempi nel dataset: {len(esempi)}')
print(f'  File: {DATASET_PATH}')
print('\n--- Esempio casuale ---')
import random
ex = random.choice(esempi)
for msg in ex['messages']:
    print(f"\n[{msg['role'].upper()}]\n{msg['content'][:300]}")

## ✏️ Editing opzionale del dataset

Puoi modificare `data/processed/{dominio}.jsonl` **prima di avviare il training**:

- Scarica il file: `Files` (icona cartella a sinistra) → naviga in `data/processed/` → tasto destro → *Download*
- Modifica con qualsiasi editor di testo (ogni riga = un esempio JSON)
- Ricarica: trascina il file modificato nella stessa cartella in Colab

Oppure edita direttamente qui sotto con Python. Il training usa sempre il file salvato su disco.

## 5. Training QLoRA

In [ ]:
import uuid
RUN_ID = f'{DOMINIO}-{uuid.uuid4().hex[:6]}'
print(f'Run ID: {RUN_ID}')

!python -m src.cli train \
    {DATASET_PATH} \
    --dominio {DOMINIO} \
    --modello {MODELLO} \
    --run-id {RUN_ID}

In [ ]:
from pathlib import Path
merged_path = Path(f'outputs/checkpoints/{RUN_ID}/merged')
if not merged_path.exists():
    raise FileNotFoundError(f'Merged model non trovato: {merged_path}')
print(f'✓ Modello merged: {merged_path}')
print(f'  File: {[f.name for f in merged_path.iterdir()]}')

## 6. Export su HuggingFace Hub

In [ ]:
if HF_REPO:
    import subprocess, sys
    result = subprocess.run(
        [sys.executable, "-m", "src.cli", "export", merged_path, "--format", "hf", "--repo-id", HF_REPO],
        capture_output=False
    )
    if result.returncode == 0:
        print(f'\n✓ Modello disponibile su: https://huggingface.co/{HF_REPO}')
    else:
        print('\n❌ Export fallito. Leggi il messaggio di errore sopra.')
        print('   Causa più comune: HF_TOKEN non ha permessi di scrittura.')
        print('   → https://huggingface.co/settings/tokens → New token → Type: Write')
else:
    print('⚠️  HF_REPO non impostato — aggiungi nei Secrets o decommenta nella cella parametri')

## 7. Test del modello

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, yaml

tokenizer = AutoTokenizer.from_pretrained(str(merged_path))
model = AutoModelForCausalLM.from_pretrained(
    str(merged_path), torch_dtype=torch.float16, device_map='auto'
)

with open(f'config/domains/{DOMINIO}.yaml') as f:
    cfg = yaml.safe_load(f)
system_prompt = cfg.get('dataset', {}).get('obiettivo', 'Sei un assistente utile.')

def chat(user_msg):
    msgs = [{'role': 'system', 'content': system_prompt}, {'role': 'user', 'content': user_msg}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

q = 'Vorrei prenotare un tavolo per sabato sera per 4 persone.'
print(f'User: {q}\n\nAssistant: {chat(q)}')